In [0]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [0]:
from pyspark.sql.functions import col, count, when, trim, avg, desc, round

df = spark.table("workspace.default.wa_fn_use_c_telco_customer_churn")

# Menampilkan 5 record
df.limit(5).show()

+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|customerID|gender|SeniorCitizen|Partner|Dependents|tenure|PhoneService|   MultipleLines|InternetService|OnlineSecurity|OnlineBackup|DeviceProtection|TechSupport|StreamingTV|StreamingMovies|      Contract|PaperlessBilling|       PaymentMethod|MonthlyCharges|TotalCharges|Churn|
+----------+------+-------------+-------+----------+------+------------+----------------+---------------+--------------+------------+----------------+-----------+-----------+---------------+--------------+----------------+--------------------+--------------+------------+-----+
|7590-VHVEG|Female|            0|    Yes|        No|     1|          No|No phone service|            DSL|            No|         Yes|              No|         No|    

In [0]:
# 1. Menampilkan Skema
print("--- Skema Data ---")
df.printSchema()

# 2. Menghitung jumlah total baris
total_rows = df.count()
print(f"\n--- Jumlah Total Baris ---")
print(f"Dataset ini memiliki {total_rows} baris.")

--- Skema Data ---
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: long (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: long (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)


--- Jumlah Total Baris ---
Dataset ini memiliki 7043 baris.


In [0]:
# Membuat query dinamis untuk menghitung 'NULL' ATAU 'spasi kosong' di setiap kolom
null_check_exprs = [
    count(when(col(c).isNull() | (trim(col(c)) == ''), c)).alias(c) 
    for c in df.columns
]

print("--- Pengecekan Nilai Kosong (Null atau ' ') ---")
display(df.select(null_check_exprs))

--- Pengecekan Nilai Kosong (Null atau ' ') ---


customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11,0


In [0]:
# Memilih kolom numerik yang relevan
numeric_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

print("--- Statistik Deskriptif ---")
display(df.describe(numeric_cols))

--- Statistik Deskriptif ---


summary,tenure,MonthlyCharges,TotalCharges
count,7043,7043,7043
mean,32.37114865824223,64.76169246059922,2283.300440841867
stddev,24.55948102309448,30.09004709767847,2266.7713618831467
min,0,18.25,
max,72,118.75,999.9


In [0]:
print("--- Distribusi Kolom Target (Churn) ---")

# Analisis churn
churn_distribution = (df.groupBy('Churn')
   .count()
   .withColumn('percentage', round((col('count') / total_rows) * 100, 2))
)

display(churn_distribution)

--- Distribusi Kolom Target (Churn) ---


Churn,count,percentage
No,5174,73.46
Yes,1869,26.54


In [0]:
# Membuat temporary view
df.createOrReplaceTempView("churn_data")

In [0]:
%sql
-- Melihat tingkat churn berdasarkan tipe kontrak
SELECT
  Contract,
  COUNT(*) AS jumlah_pelanggan,
  AVG(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0.0 END) AS tingkat_churn
FROM
  churn_data
GROUP BY
  Contract
ORDER BY
  tingkat_churn DESC

Contract,jumlah_pelanggan,tingkat_churn
Month-to-month,3875,0.42710
One year,1473,0.11270
Two year,1695,0.02832


In [0]:
%sql
-- Melihat tingkat churn berdasarkan jenis Layanan Internet
SELECT
  InternetService,
  COUNT(*) AS jumlah_pelanggan,
  AVG(CASE WHEN Churn = 'Yes' THEN 1.0 ELSE 0.0 END) AS tingkat_churn
FROM
  churn_data
GROUP BY
  InternetService
ORDER BY
  tingkat_churn DESC

InternetService,jumlah_pelanggan,tingkat_churn
Fiber optic,3096,0.41893
DSL,2421,0.18959
No,1526,0.07405


In [0]:
%sql
-- Melihat hubungan antara fitur numerik dengan churn
SELECT
  Churn,
  AVG(tenure) AS Rata2_Lama_Langganan,
  AVG(MonthlyCharges) AS Rata2_Biaya_Bulanan,
  AVG(try_cast(TotalCharges AS DOUBLE)) AS Rata2_Total_Tagihan
FROM
  churn_data
GROUP BY
  Churn

Churn,Rata2_Lama_Langganan,Rata2_Biaya_Bulanan,Rata2_Total_Tagihan
No,37.56996521066873,61.2651236953999,2555.3441410032997
Yes,17.979133226324237,74.4413322632423,1531.7960941680035
